In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel

ANES_YEAR_MIN = 1972 # Lower limit of years to consider for ANES-based model

anes_full = pd.read_csv('raw_data/anes_timeseries_cdf_csv_20260205/anes_timeseries_cdf_csv_20260205.csv')
anes_full = anes_full.rename(columns={
    "VCF0803": "voter_ideo", #Ideological self-placement scale
    "VCF9096": "gop_ideo", #Respondent rating of gop candidate on ideology scale
    "VCF9088": "dem_ideo", #Respondentt rating of dem candidate on ideology scale
    "VCF0830": "voter_blacks", #Government aid to blacks question, selected due to time continuity from 1972-2024
    "VCF9092": "gop_blacks", #Respondent rating of gop candidate on gov't aid to blacks
    "VCF9084": "dem_blacks", #Respondent rating of dem candidate on gov't aid to blacks
    "VCF0901a": "state_fips",
    "VCF0004": "year"})
anes_full['state_fips'] = pd.to_numeric(anes_full['state_fips'], errors='coerce').astype('Int64')

# Merge with state postal codes (for interpretability) and region codes
state_codes = pd.read_csv('raw_data/state_fips.csv')
anes_full = pd.merge(anes_full, state_codes, on='state_fips', how='left')
regions = pd.read_csv('raw_data/us_regions_divisions.csv')
regions = regions.rename(columns = {'State': 'state_name', 'State Code': 'state', 'Region': 'census_region', 'Division': 'division'})
anes_full = pd.merge(anes_full, regions[['state','census_region','division']], on = 'state', how = 'left')
bea_regions = pd.read_csv('raw_data/statelevel_predictors.csv')
anes_full = pd.merge(anes_full, bea_regions[['state','bea_region','pol_south']], on = 'state', how = 'left')

# Add state partisanship data to better predict distribution of ideology within states
state_partisanship = pd.read_stata('raw_data/1868_2020_presvote.dta')
state_partisanship['natl_rvote'] = (state_partisanship.groupby('year')['rvote']
        .transform(lambda x: x[state_partisanship.loc[x.index, 'state'] == 'US'].iloc[0]))
state_partisanship['rlean'] = state_partisanship['rvote'] - state_partisanship['natl_rvote']
state_partisanship['rlean_rolling2'] = (state_partisanship.groupby('state')['rlean']
        .rolling(window = 2, min_periods = 1).mean().reset_index(level=0, drop=True) 
)
state_partisanship = state_partisanship[state_partisanship['year'] >= ANES_YEAR_MIN]
anes_full = pd.merge(anes_full, state_partisanship[['year','state','rlean','rlean_rolling2']], on = ['state', 'year'], how = 'left')
anes_full['rlean'] = anes_full.groupby('state')['rlean'].ffill() #fill missing values in mid-election year with rlean from previous year
anes_full['rlean_rolling2'] = anes_full.groupby('state')['rlean_rolling2'].ffill()
# TODO: Extract 'totev' from the state partisanship data to model electoral votes

# Join mass economic and policy liberalism measures from Caughey Warshaw 2018 with ANES respondent ideology data
caughey_warshaw = pd.read_stata('/Users/marcetter/Dropbox/writing_sample/caughey_warshaw_2018_replication/caughey_warshaw_summary.dta')
caughey_warshaw = caughey_warshaw.rename(columns = {'stpo': 'state'})
caughey_warshaw['year'] = caughey_warshaw['year'].astype('int64')
anes_full = pd.merge(anes_full, caughey_warshaw[['year','state','masseconlib_est','masssociallib_est']], on = ['state','year'], how = 'left')
#extrapolate most recent estimates in 2014 to 2024
anes_full['masseconlib_est'] = anes_full.groupby('state')['masseconlib_est'].ffill()
anes_full['masssociallib_est'] = anes_full.groupby('state')['masssociallib_est'].ffill()

anes_full['voter_ideo'] = pd.to_numeric(anes_full['voter_ideo'], errors='coerce')
anes_full['voter_ideo'] = anes_full['voter_ideo'].apply(lambda x: np.nan if x == 9.0 or x == 0.0 else x)
anes_full['voter_blacks'] = pd.to_numeric(anes_full['voter_blacks'], errors='coerce')
anes_full['voter_blacks'] = anes_full['voter_blacks'].apply(lambda x: np.nan if x == 9.0 or x == 0.0 else x)

## Create predictor variables for ordinal model
anes_full['state_fips'] = anes_full['state_fips'].astype('string')
anes_full['decade'] = ((anes_full['year'] // 10) * 10).astype('string')
#anes['year_categorical'] = anes['year'].astype('category')
anes_full['year_center'] = anes_full['year'] - 2000
anes_full['year_center_sq'] = anes_full['year_center']**2

anes = anes_full[[
    'state','state_fips',
    'voter_ideo','voter_blacks',
    'year', 'year_center', 'year_center_sq', 'decade',
    'census_region','division','bea_region',
    'masseconlib_est', 'masssociallib_est',
    'rlean','rlean_rolling2']]
anes = anes[anes['year'] >= ANES_YEAR_MIN]

<positron-console-cell-83>:9: DtypeWarning: Columns (0: VCF0009x, 1: VCF0010x, 2: VCF0011x, 3: VCF0009y, 4: VCF0010y, 5: VCF0011y, 6: VCF9999, 7: VCF0012, 8: VCF0015a, 9: VCF0015b, 10: VCF0016, 11: VCF0018a, 12: VCF0018b, 13: VCF0019, 14: VCF0050a, 15: VCF0050b, 16: VCF0070a, 17: VCF0070b, 18: VCF0071a, 19: VCF0071b, 20: VCF0071c, 21: VCF0071d, 22: VCF0072a, 23: VCF0072b, 24: VCF0101, 25: VCF0105a, 26: VCF0105b, 27: VCF0106, 28: VCF0107, 29: VCF0108, 30: VCF0109, 31: VCF0111, 32: VCF0112, 33: VCF0113, 34: VCF0114, 35: VCF0115, 36: VCF0116, 37: VCF0117, 38: VCF0118, 39: VCF0119, 40: VCF0120, 41: VCF0121, 42: VCF0122, 43: VCF0123, 44: VCF0124, 45: VCF0125, 46: VCF0126, 47: VCF0126a, 48: VCF0126b, 49: VCF0126c, 50: VCF0127, 51: VCF0127a, 52: VCF0127b, 53: VCF0128, 54: VCF0128a, 55: VCF0128b, 56: VCF0129, 57: VCF0130, 58: VCF0130a, 59: VCF0131, 60: VCF0132, 61: VCF0133, 62: VCF0134, 63: VCF0135, 64: VCF0136, 65: VCF0137, 66: VCF0138, 67: VCF0138a, 68: VCF0138b, 69: VCF0138c, 70: VCF0138d, 

In [ ]:

ideo_data = anes[['voter_ideo', 'state', 
    'rlean_rolling2', 'masssociallib_est', 'masseconlib_est']].dropna().copy()
ideo_model = OrderedModel.from_formula('voter_ideo ~ state + masseconlib_est + masssociallib_est + rlean_rolling2', 
    distr = 'logit', data = ideo_data, hasconst=False).fit(method = 'cg', disp = True)
ideo_model.summary() #switching to method 'cg' (conjugate gradient) from default Nelder Mead resolves convergence warnings


Optimization terminated successfully.
         Current function value: 1.732836
         Iterations: 329
         Function evaluations: 636
         Gradient evaluations: 636


<class 'statsmodels.iolib.summary.Summary'>
"""
                             OrderedModel Results                             
==============================================================================
Dep. Variable:             voter_ideo   Log-Likelihood:                -72531.
Model:                   OrderedModel   AIC:                         1.452e+05
Method:            Maximum Likelihood   BIC:                         1.457e+05
Date:                Fri, 07 Aug 2026                                         
Time:                        23:38:26                                         
No. Observations:               41857                                         
Df Residuals:                   41799                                         
Df Model:                          52                                         
=====================================================================================
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
state[T.AL]           0.2140      0.512      0.418      0.676      -0.789       1.217
state[T.AR]           0.1958      0.513      0.382      0.703      -0.809       1.201
state[T.AZ]           0.0464      0.511      0.091      0.928      -0.956       1.048
state[T.CA]          -0.0196      0.509     -0.038      0.969      -1.018       0.979
state[T.CO]          -0.2126      0.511     -0.416      0.677      -1.214       0.789
state[T.CT]          -0.0162      0.514     -0.032      0.975      -1.024       0.992
state[T.DE]          -0.1770      0.549     -0.322      0.747      -1.254       0.900
state[T.FL]           0.1451      0.509      0.285      0.776      -0.853       1.143
state[T.GA]           0.1332      0.510      0.261      0.794      -0.866       1.132
state[T.HI]           0.0243      0.573      0.042      0.966      -1.099       1.147
state[T.IA]           0.1173      0.513      0.229      0.819      -0.887       1.122
state[T.ID]          -0.2315      0.529     -0.438      0.662      -1.269       0.806
state[T.IL]          -0.0338      0.510     -0.066      0.947      -1.034       0.966
state[T.IN]           0.0889      0.510      0.174      0.862      -0.911       1.089
state[T.KS]           0.1466      0.513      0.286      0.775      -0.859       1.152
state[T.KY]          -0.3002      0.515     -0.582      0.560      -1.310       0.710
state[T.LA]           0.1306      0.514      0.254      0.800      -0.877       1.139
state[T.MA]          -0.1201      0.513     -0.234      0.815      -1.126       0.886
state[T.MD]           0.1151      0.513      0.224      0.822      -0.890       1.120
state[T.ME]          -0.1494      0.527     -0.284      0.777      -1.182       0.883
state[T.MI]           0.0484      0.510      0.095      0.924      -0.951       1.048
state[T.MN]           0.0602      0.511      0.118      0.906      -0.941       1.061
state[T.MO]           0.0241      0.510      0.047      0.962      -0.976       1.024
state[T.MS]           0.3501      0.519      0.675      0.500      -0.667       1.367
state[T.MT]          -0.2620      0.557     -0.470      0.638      -1.354       0.830
state[T.NC]           0.0233      0.510      0.046      0.964      -0.976       1.023
state[T.ND]           0.2451      0.538      0.456      0.649      -0.809       1.299
state[T.NE]          -0.0874      0.519     -0.168      0.866      -1.105       0.930
state[T.NH]          -0.1256      0.518     -0.242      0.809      -1.141       0.890
state[T.NJ]           0.0896      0.512      0.175      0.861      -0.914       1.093
state[T.NM]           0.1440      0.521      0.277      0.782      -0.876       1.164
state[T.NV]           0.0573      0.528      0.109      0.914      -0.978       1.092
state[T.NY]           0.0533      0.512      0.104      0.917      -0.949       1.056
state[T.OH]           0.0792      0.509      0.156      0.876     

In [ ]:
YEAR = 2000
masslib = anes[['year','state','masseconlib_est','masssociallib_est']].drop_duplicates()
masslib = masslib.set_index(['state','year'], drop = False)
"""
(ideo_model.predict(pd.DataFrame({
    'state': ['CA','CT','VT'], 

    'masseconlib_est': [masslib.loc[('CA',YEAR),'masseconlib_est'],
        masslib.loc[('CT',YEAR),'masseconlib_est'],
        masslib.loc[('VT',YEAR),'masseconlib_est']],
    'masssociallib_est': [masslib.loc[('CA',YEAR),'masssociallib_est'], 
        masslib.loc[('CT',YEAR),'masssociallib_est'],
        masslib.loc[('VT',YEAR),'masssociallib_est']],

    'year_center': [-20,-20,-20],
    'bea_region': ['Southeast', 'Far West', 'Mideast'], 
    'decade': ['2000','2020','1980'], 
    'rlean_rolling2': [15.5,10.5,-5.5]}))
    .cumsum(axis = 'columns')
)
"""


(ideo_model.predict(pd.DataFrame({
    'state': ['CA','CT','VT'], 
    'rlean_rolling2': [-10.5,-6.7,-8.9],
    'masseconlib_est': [1,0.5,0.8],
    'masssociallib_est': [1,0.5,0.8]}))
    .cumsum(axis = 'columns')
)

#TODO: Create table containing covariates of each state for every election year from 1972-2024 to predict
# distribution of self-placement in each state. Plot average public opinion in each state to visually assess
# estimates

,0,1,2,3,4,5,6
0,0.059320,0.260484,0.426305,0.742604,0.863730,0.978267,1.0
1,0.044110,0.204931,0.352232,0.678579,0.822638,0.970535,1.0
2,0.087291,0.348199,0.529852,0.813974,0.905776,0.985563,1.0


In [ ]:
resent_data = anes[['voter_blacks', 'state', 
    'rlean_rolling2', 'masssociallib_est', 'masseconlib_est']].dropna().copy()
resent_model = OrderedModel.from_formula('voter_blacks ~ state + masseconlib_est + masssociallib_est + rlean_rolling2', 
    distr = 'logit', data = resent_data, hasconst=False).fit(method = 'cg', disp = True)
resent_model.summary() #switching to method 'cg' (conjugate gradient) from default Nelder Mead resolves convergence warnings


Optimization terminated successfully.
         Current function value: 1.873410
         Iterations: 287
         Function evaluations: 603
         Gradient evaluations: 603


<class 'statsmodels.iolib.summary.Summary'>
"""
                             OrderedModel Results                             
==============================================================================
Dep. Variable:           voter_blacks   Log-Likelihood:                -86967.
Model:                   OrderedModel   AIC:                         1.741e+05
Method:            Maximum Likelihood   BIC:                         1.746e+05
Date:                Sat, 08 Aug 2026                                         
Time:                        00:00:41                                         
No. Observations:               46422                                         
Df Residuals:                   46364                                         
Df Model:                          52                                         
=====================================================================================
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
state[T.AL]           0.0291      0.526      0.055      0.956      -1.001       1.060
state[T.AR]           0.2652      0.526      0.504      0.614      -0.766       1.297
state[T.AZ]           0.2217      0.526      0.422      0.673      -0.809       1.252
state[T.CA]          -0.0761      0.524     -0.145      0.885      -1.103       0.951
state[T.CO]          -0.0013      0.525     -0.003      0.998      -1.031       1.028
state[T.CT]          -0.2072      0.528     -0.393      0.695      -1.241       0.827
state[T.DE]          -0.1733      0.562     -0.309      0.758      -1.274       0.927
state[T.FL]           0.1095      0.524      0.209      0.834      -0.917       1.136
state[T.GA]          -0.0760      0.524     -0.145      0.885      -1.104       0.951
state[T.HI]           0.4151      0.589      0.704      0.481      -0.740       1.570
state[T.IA]           0.0610      0.526      0.116      0.908      -0.971       1.093
state[T.ID]          -0.1909      0.543     -0.351      0.725      -1.255       0.874
state[T.IL]          -0.1099      0.525     -0.209      0.834      -1.138       0.918
state[T.IN]           0.1293      0.524      0.247      0.805      -0.898       1.156
state[T.KS]           0.0838      0.527      0.159      0.874      -0.950       1.117
state[T.KY]           0.0464      0.529      0.088      0.930      -0.990       1.083
state[T.LA]           0.0417      0.528      0.079      0.937      -0.993       1.076
state[T.MA]          -0.1783      0.527     -0.338      0.735      -1.212       0.855
state[T.MD]          -0.0252      0.527     -0.048      0.962      -1.058       1.008
state[T.ME]          -0.2981      0.538     -0.554      0.579      -1.352       0.756
state[T.MI]          -0.0386      0.524     -0.074      0.941      -1.066       0.989
state[T.MN]           0.0881      0.525      0.168      0.867      -0.941       1.117
state[T.MO]           0.1286      0.525      0.245      0.807      -0.900       1.157
state[T.MS]          -0.0590      0.532     -0.111      0.912      -1.101       0.984
state[T.MT]           0.2084      0.567      0.368      0.713      -0.902       1.319
state[T.NC]          -0.1127      0.524     -0.215      0.830      -1.140       0.915
state[T.ND]           0.4699      0.554      0.848      0.396      -0.616       1.556
state[T.NE]           0.3441      0.531      0.648      0.517      -0.696       1.384
state[T.NH]          -0.1112      0.531     -0.209      0.834      -1.153       0.930
state[T.NJ]           0.0765      0.526      0.145      0.884      -0.955       1.108
state[T.NM]           0.0610      0.534      0.114      0.909      -0.985       1.107
state[T.NV]           0.2847      0.541      0.526      0.599      -0.775       1.345
state[T.NY]          -0.2035      0.526     -0.387      0.699      -1.234       0.827
state[T.OH]           0.0830      0.524      0.159      0.874     

In [ ]:
(resent_model.predict(pd.DataFrame({
    'state': ['AK','MS','NJ'],  
    'rlean_rolling2': [15.5,16.2,-5],
    'masseconlib_est': [0.5,0.3,1.2],
    'masssociallib_est': [0.5,0.3,1.2]}))
    .cumsum(axis = 'columns')
)

#TODO: Create table containing covariates of each state for every election year from 1972-2024 to predict
# distribution of self-placement in each state. Plot average public opinion in each state to visually assess
# estimates

,0,1,2,3,4,5,6
0,0.075711,0.131678,0.222380,0.452664,0.602566,0.759864,1.0
1,0.081317,0.140797,0.236071,0.471929,0.620972,0.773721,1.0
2,0.084751,0.146342,0.244302,0.483182,0.631528,0.781520,1.0


46570

In [ ]:
"""Load the greater US shapefile data in the working directory under greater_us_shapefiles/ and 
create three plots:
1)
2)
3) """ ""


states = gpd.read_file("path/to/us_states.shp")
states = states.rename(columns={"STUSPS": "state"})

# Keep only the variables needed for plotting
states = states[["state", "geometry"]]

# Merge geometry with ideology estimates
plot_data = states.merge(ideology,
    on="state",
    how="left"
)
decades = np.arange(
    ideology["year"].min() // 10 * 10,
    ideology["year"].max() + 1,
    10
)

plot_data = plot_data[
    plot_data["year"].isin(decades)
]
ncols = 3
nrows = int(np.ceil(len(decades) / ncols))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(15, 4.5 * nrows)
)

# Make axes iterable even if there is only one panel
axes = np.atleast_1d(axes).flatten()
